In [1]:
!pip install -q transformers torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/DialoGPT-medium"

print("Loading chatbot model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

tokenizer.pad_token = tokenizer.eos_token
model.eval()

print("Model loaded successfully!")
print("Device:", device)
print("\nChatbot: Hello! Type 'exit' to end the conversation.\n")

chat_history_ids = None

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break

    if not user_input:
        print("Chatbot: Please enter a message.")
        continue

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    if chat_history_ids is None:
        bot_input_ids = new_input_ids
    else:
        bot_input_ids = torch.cat(
            [chat_history_ids, new_input_ids],
            dim=-1
        )

    if bot_input_ids.shape[-1] > 800:
        bot_input_ids = bot_input_ids[:, -800:]

    attention_mask = torch.ones_like(bot_input_ids)

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response_ids = chat_history_ids[:, bot_input_ids.shape[-1]:]

    response = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    if response:
        print("Chatbot:", response)
    else:
        print("Chatbot: I am not sure how to respond to that.")

Loading chatbot model...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  863MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda

Chatbot: Hello! Type 'exit' to end the conversation.

You: Hi
Chatbot: Hey ! How are you ?
You: exit
Chatbot: Goodbye! Have a nice day.
